# 把 PyBullet 渲染搬上 GPU：在真实 episode 上验证三步计划

<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/notebooks/pybullet_gpu_pipeline_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **这个 notebook 现在跑不起来了，这是有意的。**
> 它对比的"方法 A"用的是 `TensorRobotRenderer`（yourdfpy），而它得出的结论
> 正是把那条路径换掉——所以那个类已经从 `core/physics.py` 里删除了。
> 第 1、2 节（`gpu=True` 的等价性、渲染分辨率）仍然可跑；第 3 步的 A/B 需要
> `git checkout 6f607fa` 回到改动前的代码。结论和全部数字都保留在下面的表里。

前一个 notebook（[pybullet_egl_mask_benchmark.ipynb](pybullet_egl_mask_benchmark.ipynb)）
在一个合成 pose 上证明了：EGL 光栅化器快 4.4 倍，但 `alpha=0` 隐藏在它下面失效，
必须改成从 URDF 里删几何。这个 notebook 拿**真实 DROID episode** 验证三件事：

| | 主张 | 怎么算验过 |
|---|---|---|
| **第 1 步** | `PyBulletRenderer(gpu=True)` 在真实数据上与 CPU 等价且更快 | 三台相机、多帧的 mask IoU / 深度分位数 / 每帧耗时 |
| **第 2 步** | 点云生成可以降分辨率，换更大的加速 | 各分辨率下的点数、耗时，以及世界点云是否还是同一朵 |
| **第 3 步** | 基于 PyBullet 的外参优化可以取代 yourdfpy | 同一 episode 上两条路各跑一遍，用 `compute_metrics.evaluate_extrinsics` 这把中立尺子量 |

三步是有依赖顺序的：第 2 步依赖第 1 步的 `gpu=True`，第 3 步依赖前两步。
所以下面按顺序走，前一步不过关就没必要看后一步。

---

### 跑之前

**1. 需要 GPU runtime**，否则 EGL 加载不上，全部退回 CPU，对照不成立。第 0 节会明说。

**2. 需要一个真实 episode 的 Stage 1 输出**（depth + robot + calibration）。
本地 checkout 会直接用 `data/cache/depth/`；Colab 上从
`gs://dm-tapnet/tmp/droid/` 拉一个 episode（需要 `gcloud` 认证，约 1-2 GB）。

**3. 会编译一次 PyBullet**（要 NumPy 支持），Colab 上十几分钟。已装好则跳过。

## 0. 环境

In [ ]:
import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/droid"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/yangyi02/droid.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()
    while not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
        parent = os.path.dirname(REPO_DIR)
        if parent == REPO_DIR:
            REPO_DIR = os.getcwd()
            break
        REPO_DIR = parent

if not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
    raise SystemExit(f"{REPO_DIR} is not the droid repo root -- open this "
                     "notebook from the checkout, or set REPO_DIR by hand.")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")
print(f"{'Colab' if IN_COLAB else 'Local'}: {REPO_DIR}")

In [ ]:
import subprocess
import sys

def numpy_enabled():
    out = subprocess.run([sys.executable, "-c",
                          "import pybullet as p; print(p.isNumpyEnabled())"],
                         capture_output=True, text=True)
    return out.stdout.strip().endswith("1")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "yourdfpy", "trimesh", "mediapy"], check=False)

if numpy_enabled():
    print("pybullet already has NumPy support")
else:
    print("Rebuilding pybullet from source -- minutes.")
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
         "--no-binary", "pybullet", "--no-build-isolation", "--no-cache-dir", "pybullet"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line for k in ("numpy is", "Building wheel", "Successfully", "error")):
            print("   ", line.rstrip())
    print("   isNumpyEnabled =", numpy_enabled())
    if IN_COLAB:
        print("\n>>> Colab 需要重启运行时（代码执行程序 → 重新启动会话）再从这里继续。")

In [ ]:
import importlib.util

import pybullet as p
import torch

assert p.isNumpyEnabled(), "上一格装完了吗？Colab 上可能需要先重启运行时"

p.connect(p.DIRECT)
spec = importlib.util.find_spec("eglRenderer")
EGL_OK = spec is not None and p.loadPlugin(spec.origin, "_eglRendererPlugin") >= 0
p.disconnect()

print("torch.cuda        :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("EGL plugin loads  :", EGL_OK)
if not EGL_OK:
    print("\n没有 GPU 光栅化器，第 1-3 步的 gpu=True 会静默退回 CPU，结论不成立。"
          "\nColab: 代码执行程序 → 更改运行时类型 → GPU，然后从头再跑。")

## 1. 数据：一个真实 episode

用仓库自己的 loader (`core.io.load_depth_data` / `load_extrinsics`)，
拿到的就是 `compute_tracks` / `compute_metrics` 平时吃的那份 `scene_constants`。

这里跑的是整个 episode 的每一帧，跟 `compute_extrinsics` 吃的是同一批帧。

In [ ]:
import glob

import numpy as np

EPISODE_ID = ""

DEPTH_ROOT = os.path.join(CACHE_DIR, "depth")
EXT_ROOT = os.path.join(CACHE_DIR, "extrinsics")
GCS = "gs://dm-tapnet/tmp/droid"

local = sorted(d for d in glob.glob(os.path.join(DEPTH_ROOT, "*"))
               if os.path.exists(os.path.join(d, "robot.npz")))
if not EPISODE_ID:
    if local:
        EPISODE_ID = os.path.basename(local[0])
    elif IN_COLAB:
        EPISODE_ID = "TRI+52ca9b6a+2023-12-13-14h-03m-35s"
    else:
        raise SystemExit(f"No episodes under {DEPTH_ROOT} -- set EPISODE_ID and "
                         "let the next cell fetch it from GCS.")
print("episode:", EPISODE_ID)

ep_cache = os.path.join(DEPTH_ROOT, EPISODE_ID)
if not os.path.exists(os.path.join(ep_cache, "robot.npz")):
    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
    os.makedirs(ep_cache, exist_ok=True)
    os.system(f"gsutil -m cp -r '{GCS}/depth/{EPISODE_ID}/*' '{ep_cache}/'")
    os.makedirs(os.path.join(EXT_ROOT, EPISODE_ID), exist_ok=True)
    os.system(f"gsutil -m cp -r '{GCS}/extrinsics/{EPISODE_ID}/*' "
              f"'{os.path.join(EXT_ROOT, EPISODE_ID)}/'")
print("cached at:", ep_cache)

In [ ]:
import torch

import core.io

SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)

device = core.io.get_accelerator()
scene_constants = core.io.load_depth_data(
    EPISODE_ID, DEPTH_ROOT, load_video="first_frame")
scene_state_ds = core.io.load_extrinsics(scene_constants, EXT_ROOT)

WRIST = scene_constants["meta"]["wrist_serial"]
EXT_CAMS = [c for c in scene_constants["camera"] if c != WRIST]
N_FRAMES = len(scene_constants["robot"]["joint_positions"])
H_IMG, W_IMG = scene_constants["camera"][EXT_CAMS[0]]["raw_depth"][0].shape

print(f"\n帧数     : {N_FRAMES}")
print(f"分辨率   : {W_IMG}x{H_IMG}")
print(f"外部相机 : {EXT_CAMS}")
print(f"腕部相机 : {WRIST}")

## 2. 第一步：`PyBulletRenderer(gpu=True)` 在真实数据上等价吗

合成 pose 上量到 mask IoU 0.983。真实数据的相机位姿、机器人构型、遮挡关系都不一样，
所以这里在**三台相机 × 多帧真实位姿**上重量一遍。

判据有两条，缺一不可：
- **等价**：mask IoU 要接近 1，深度在真实表面上要亚毫米一致；
- **更快**：每帧耗时要真的降下来。

In [ ]:
import time

import core.physics

N_AB = min(N_FRAMES, 12)
FRAMES_AB = np.linspace(0, N_FRAMES - 1, N_AB).astype(int)

def render_all(gpu):
    r = core.physics.PyBulletRenderer(gpu=gpu)
    out, per_frame_ms = {}, []
    for cam in scene_constants["camera"]:
        K = scene_constants["camera"][cam]["K_mat"]
        for t in FRAMES_AB:
            scene_constants["robot"]["joint_positions"][t]
            r.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                scene_constants["robot"]["gripper_positions"][t])
            ext = scene_state_ds[cam]["extrinsics"][t]
            t0 = time.perf_counter()
            d = r.render_depth(ext, K, W_IMG, H_IMG)
            per_frame_ms.append((time.perf_counter() - t0) * 1000)
            out[(cam, t)] = (d, r.render_mask(ext, K, W_IMG, H_IMG))
    return r.gpu, out, float(np.median(per_frame_ms))

gpu_ok, RES_GPU, MS_GPU = render_all(True)
_, RES_CPU, MS_CPU = render_all(False)
assert gpu_ok, "EGL 没加载上，下面的对照没有意义"
print(f"CPU {MS_CPU:6.1f} ms/frame   GPU {MS_GPU:6.1f} ms/frame   {MS_CPU / MS_GPU:.1f}x")

In [ ]:
from scipy import ndimage

print(f"{'camera':<12}{'帧':>4}{'mask IoU':>11}{'mask 覆盖 cpu/gpu':>20}"
      f"{'表面深度中位差':>16}{'表面最大差':>13}")
rows = []
for cam in scene_constants["camera"]:
    ious, med_smooth, max_smooth, cov_c, cov_g = [], [], [], [], []
    for t in FRAMES_AB:
        dc, mc = RES_CPU[(cam, t)]
        dg, mg = RES_GPU[(cam, t)]
        union = (mc | mg).sum()
        if union == 0:
            continue
        ious.append((mc & mg).sum() / union)
        cov_c.append(mc.mean()); cov_g.append(mg.mean())
        both = (dc > 0) & (dg > 0)
        if both.sum() < 100:
            continue
        hi = ndimage.maximum_filter(np.where(dc > 0, dc, -1e9), size=3)
        lo = ndimage.minimum_filter(np.where(dc > 0, dc, +1e9), size=3)
        smooth = both & ((hi - lo) < 1e-3)
        if smooth.sum() > 100:
            diff = np.abs(dc - dg)[smooth]
            med_smooth.append(np.median(diff)); max_smooth.append(diff.max())
    tag = "wrist" if cam == WRIST else "ext"
    rows.append((cam, tag, np.mean(ious), np.mean(cov_c), np.mean(cov_g),
                 np.median(med_smooth), np.max(max_smooth)))
    print(f"{cam:<12}{len(ious):>4}{np.mean(ious):>11.4f}"
          f"{100 * np.mean(cov_c):>10.2f}% /{100 * np.mean(cov_g):>6.2f}%"
          f"{np.median(med_smooth):>14.2e} m{np.max(max_smooth):>11.2e} m")
print(f"\n每帧耗时: CPU {MS_CPU:.1f} ms  ->  GPU {MS_GPU:.1f} ms  ({MS_CPU / MS_GPU:.1f}x)")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage

C_CPU, C_GPU, C_THIRD = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"

cam_show = EXT_CAMS[0]
t_show = 0
rgb = scene_constants["camera"][cam_show].get("first_frame_rgb")
_, mc = RES_CPU[(cam_show, t_show)]
_, mg = RES_GPU[(cam_show, t_show)]

ys, xs = np.where(mc | mg)
box = (slice(max(ys.min() - 30, 0), ys.max() + 30),
       slice(max(xs.min() - 30, 0), xs.max() + 30))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=SURFACE)
base = (rgb[box] / 255.0 if rgb is not None
        else np.ones((box[0].stop - box[0].start, box[1].stop - box[1].start, 3)) * 0.9)
for ax, (m, label, colour) in zip(axes, [(mc[box], "CPU rasteriser", C_CPU),
                                         (mg[box], "EGL rasteriser (gpu=True)", C_GPU),
                                         (None, "disagreement", C_THIRD)]):
    img = base.copy()
    if m is None:
        d = mc[box] ^ mg[box]
        n_px = int(d.sum())
        img[ndimage.binary_dilation(d, iterations=2)] = matplotlib.colors.to_rgb(C_THIRD)
        label = f"disagreement: {n_px} px (dilated 2px to be visible)"
    else:
        img[m] = 0.35 * img[m] + 0.65 * np.array(matplotlib.colors.to_rgb(colour))
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(label, fontsize=9.5, color=INK, pad=8)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle(f"{cam_show}, frame {t_show} -- same pose, two rasterisers",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
plt.show()

## 3. 第二步：点云生成能降到多低的分辨率

外参优化只需要 `MAX_ROBOT_PTS`（仓库里是 2000）个机器人表面点，
但 `get_foreground_robot_points` 是在**全分辨率**渲染完再丢掉 95%。
降分辨率只要把 `K` 同比缩放，反投影出的世界点坐标不变。

两个问题：**点还够不够**，以及**点云是不是同一朵**。

（下面量到的结论是"降到一半误差还埋在采样噪声里"，但流水线**最终没有采用**：
`compute_extrinsics` 一律全分辨率渲染，少一个能让某些 episode 取不到点云而
直接报错的旋钮。这一节保留下来是记录降分辨率能省多少。）

In [ ]:
import compute_metrics

MAX_ROBOT_PTS = 2000
SCALES = [1.0, 0.75, 0.5, 0.25, 0.125]
cam = EXT_CAMS[0]
K_full = scene_constants["camera"][cam]["K_mat"]

def scaled(K, s):
    K2 = K.copy()
    K2[:2] *= s
    return K2

r_gpu = core.physics.PyBulletRenderer(gpu=True)
SWEEP = {}
print(f"{'scale':>7}{'render':>12}{'robot px':>11}{'>=2000?':>9}{'ms/frame':>11}")
for s in SCALES:
    w, h = int(W_IMG * s), int(H_IMG * s)
    K_s = scaled(K_full, s)
    counts, times = [], []
    for t in FRAMES_AB[:6]:
        r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                scene_constants["robot"]["gripper_positions"][t])
        ext = scene_state_ds[cam]["extrinsics"][t]
        r_gpu.render_depth(ext, K_s, w, h)
        t0 = time.perf_counter()
        d = r_gpu.render_depth(ext, K_s, w, h)
        times.append((time.perf_counter() - t0) * 1000)
        counts.append(int((d > 0).sum()))
    SWEEP[s] = (np.median(counts), np.median(times), w, h)
    ok = "yes" if np.median(counts) >= MAX_ROBOT_PTS else "NO"
    print(f"{s:>7.3f}{f'{w}x{h}':>12}{int(np.median(counts)):>11}{ok:>9}"
          f"{np.median(times):>11.2f}")

In [ ]:
import torch

cam = EXT_CAMS[0]
t = FRAMES_AB[len(FRAMES_AB) // 2]
r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                        scene_constants["robot"]["gripper_positions"][t])
ext = scene_state_ds[cam]["extrinsics"][t]
obs = scene_constants["camera"][cam]["raw_depth"][t].astype(np.float32)

ref = core.physics.get_foreground_robot_points(
    ext, K_full, obs, r_gpu, device, max_pts=8000)
print(f"{'scale':>7}{'点数':>8}{'到全分辨率点云的最近邻距离':>30}")
for s in SCALES:
    w, h = int(W_IMG * s), int(H_IMG * s)
    pts = core.physics.get_foreground_robot_points(
        ext, scaled(K_full, s), np.zeros((h, w), np.float32), r_gpu,
        device, max_pts=MAX_ROBOT_PTS)
    if pts is None:
        print(f"{s:>7.3f}{'--':>8}   这一帧点数不足 {MAX_ROBOT_PTS}，跳过")
        continue
    nn = torch.cdist(pts[None], ref[None])[0].min(dim=1)[0]
    print(f"{s:>7.3f}{len(pts):>8}   中位 {nn.median().item() * 1000:6.2f} mm"
          f" | p99 {torch.quantile(nn, 0.99).item() * 1000:6.2f} mm")

## 4. 第三步：外参优化，yourdfpy vs PyBullet

现在两条路都在同一个 episode 上跑一遍，从**同一个初始外参**（数据集自带的）出发：

| | 点云来源 | 可见性 | 损失 |
|---|---|---|---|
| **A（现状）** | `TensorRobotRenderer`：yourdfpy FK + mesh 表面采样 | 靠法线做 front-face culling 近似 | `compute_extrinsics.compute_robot_loss`（带法线、带 tolerance） |
| **B（复活）** | `PyBulletRenderer`：光栅化后反投影 | 光栅化器天然只给可见面 | `compute_metrics.compute_robot_loss_batched`（无法线、无 culling） |

两条路的损失函数不同，所以**收敛 loss 之间不可比**。裁判用第三方：
`compute_metrics.evaluate_extrinsics`——它对任何 `scene_state` 都用同一套
PyBullet 渲染 + Chamfer 来打分，三个状态（初始 / A / B）用同一把尺子。

> 注意 B 的点云来源和裁判共用一个渲染器。这不是循环论证（裁判量的是外参和**观测深度**
> 的一致性，不是和渲染的一致性），但也不是完全中立，读结论时要记得这一点。

In [ ]:
import copy
import time

import compute_extrinsics

t0 = time.perf_counter()
tensor_renderer = core.physics.TensorRobotRenderer(device=device)
state_A = compute_extrinsics.per_camera_alignment(
    scene_constants, tensor_renderer, scene_state_ds)
T_A = time.perf_counter() - t0
print(f"\n方法 A 用时 {T_A:.1f} s")

In [ ]:
import torch.optim as optim

import core.geometry

OUTER, INNER = 3, 167

def optimise_camera_pybullet(cam, pb, T_init_np):
    is_wrist = (cam == WRIST)
    K_np = scene_constants["camera"][cam]["K_mat"]
    K_t = torch.tensor(K_np, dtype=torch.float32, device=device)
    T_init_t = torch.tensor(T_init_np, dtype=torch.float32, device=device)

    d_ext = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d_ext], lr=0.001)
    loss_val = float("nan")

    for outer in range(OUTER):
        with torch.no_grad():
            T_cur = (T_init_t @ core.geometry.make_T(d_ext, device)).cpu().numpy()
        cache_X, cache_obs = [], []
        for t in range(N_FRAMES):
            pb.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                 scene_constants["robot"]["gripper_positions"][t])
            obs = scene_constants["camera"][cam]["raw_depth"][t].astype(np.float32)
            if is_wrist:
                T_cw = scene_constants["robot"]["T_ee_base_all"][t] @ T_cur
                pts = core.physics.get_foreground_gripper_points(
                    T_cw, K_np, obs, pb, device)
                if pts is None:
                    continue
                cache_X.append(torch.tensor(
                    (T_cur @ pts)[:3, :].T, dtype=torch.float32,
                    device=device))
            else:
                pts = core.physics.get_foreground_robot_points(T_cur, K_np, obs, pb, device)
                if pts is None:
                    continue
                cache_X.append(pts)
            cache_obs.append(torch.tensor(obs, dtype=torch.float32, device=device)[None])
        if not cache_X:
            print(f"    [WARN] {cam}: no points at outer {outer}")
            return T_init_np, float("nan")
        batch_X, batch_obs = torch.stack(cache_X), torch.stack(cache_obs)

        for _ in range(INNER):
            optimizer.zero_grad()
            T_opt = T_init_t @ core.geometry.make_T(d_ext, device)
            loss = (core.physics.compute_wrist_loss_batched(
                        batch_X, T_opt, K_t, batch_obs) if is_wrist else
                    core.physics.compute_robot_loss_batched(
                        batch_X, T_opt, K_t, batch_obs))
            loss.backward()
            optimizer.step()
            loss_val = loss.item()
        print(f"    outer {outer + 1}/{OUTER} | frames {len(cache_X)} | loss {loss_val:.4f}")

    with torch.no_grad():
        return (T_init_t @ core.geometry.make_T(d_ext, device)).cpu().numpy(), loss_val

t0 = time.perf_counter()
state_B = copy.deepcopy(scene_state_ds)
for cam in scene_constants["camera"]:
    print(f"  [{cam}] {'wrist' if cam == WRIST else 'external'}")
    T_fin, _ = optimise_camera_pybullet(cam, r_gpu, scene_state_ds[cam]["base_extrinsic"])
    state_B[cam]["base_extrinsic"] = T_fin
    state_B[cam]["extrinsics"] = (scene_constants["robot"]["T_ee_base_all"] @ T_fin
                                  if cam == WRIST else np.tile(T_fin, (N_FRAMES, 1, 1)))
T_B = time.perf_counter() - t0
print(f"\n方法 B 用时 {T_B:.1f} s")

In [ ]:
SCORES = {}
for name, st in (("起点（旧流水线产出）", scene_state_ds), ("A: yourdfpy", state_A),
                 ("B: pybullet", state_B)):
    SCORES[name] = compute_metrics.evaluate_extrinsics(
        scene_constants, st, device, pb_renderer=r_gpu)

keys = [("robot_loss_cam1", "robot cam1"), ("robot_loss_cam2", "robot cam2"),
        ("robot_loss_wrist", "robot wrist"), ("chamfer_total", "chamfer"),
        ("bg_overlap_pct", "bg overlap %")]
print(f"{'':<16}" + "".join(f"{lab:>14}" for _, lab in keys))
for name, m in SCORES.items():
    print(f"{name:<16}" + "".join(f"{m.get(k, float('nan')):>14.4f}" for k, _ in keys))
print(f"\n耗时  A {T_A:.1f} s   B {T_B:.1f} s   ({T_A / T_B:.2f}x)")

print("\n两条路解出来的外参差多少:")
for cam in scene_constants["camera"]:
    dT = np.linalg.inv(state_A[cam]["base_extrinsic"]) @ state_B[cam]["base_extrinsic"]
    shift = np.linalg.norm(dT[:3, 3]) * 1000
    rot = np.degrees(np.arccos(np.clip((np.trace(dT[:3, :3]) - 1) / 2, -1, 1)))
    print(f"  {cam:<12} 平移 {shift:7.2f} mm   旋转 {rot:6.3f}°")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=SURFACE)
cam = EXT_CAMS[0]
K = scene_constants["camera"][cam]["K_mat"]
rgb = scene_constants["camera"][cam]["first_frame_rgb"]
r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][0],
                        scene_constants["robot"]["gripper_positions"][0])

panels = [("prev pipeline", scene_state_ds, C_THIRD),
          ("A: yourdfpy", state_A, C_CPU),
          ("B: pybullet", state_B, C_GPU)]
masks = {lab: r_gpu.render_mask(st[cam]["extrinsics"][0], K, W_IMG, H_IMG)
         for lab, st, _ in panels}
ys, xs = np.where(np.logical_or.reduce(list(masks.values())))
box = (slice(max(ys.min() - 30, 0), ys.max() + 30),
       slice(max(xs.min() - 30, 0), xs.max() + 30))

for ax, (label, _, colour) in zip(axes, panels):
    m = masks[label][box]
    img = rgb[box] / 255.0
    edge = m ^ ndimage.binary_erosion(m, iterations=2)
    img = img.copy()
    img[edge] = matplotlib.colors.to_rgb(colour)
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(label, fontsize=9.5, color=INK, pad=8)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle(f"{cam}, frame 0 -- rendered robot outline over the real image",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.9), facecolor=SURFACE)
labels = ["prev pipeline", "A: yourdfpy", "B: pybullet"]
colours = [C_THIRD, C_CPU, C_GPU]
panels = [
    ("robot depth loss, cam1  (lower is better)",
     [SCORES[n].get("robot_loss_cam1", np.nan) for n in SCORES], "%.4f"),
    ("chamfer total  (lower is better)",
     [SCORES[n].get("chamfer_total", np.nan) for n in SCORES], "%.4f"),
    ("wall clock, seconds  (init is free)", [0.0, T_A, T_B], "%.0f s"),
]
for ax, (title, vals, fmt) in zip(axes, panels):
    y = np.arange(3)[::-1]
    ax.barh(y, vals, height=0.34, color=colours, zorder=3)
    span = max(v for v in vals if np.isfinite(v)) or 1.0
    for yi, v in zip(y, vals):
        if np.isfinite(v):
            ax.text(v + span * 0.03, yi, fmt % v, va="center", fontsize=9.5, color=INK)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9, color=MUTED)
    ax.set_title(title, fontsize=9.5, color=INK, loc="left", pad=10)
    ax.set_xlim(0, span * 1.3)
    ax.set_facecolor(SURFACE); ax.xaxis.set_visible(False)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(GRID); ax.tick_params(length=0)
fig.tight_layout()
plt.show()

## 5. 联合优化也换掉

align 只对齐机器人本体。`global_joint_alignment` 才是最终形态——
Chamfer 环境缝合 + 三台相机的机器人深度损失一起优化，三个 6-DoF 增量同时动。

它对 yourdfpy 的依赖只有一处：三次 `extract_robot_physical_tensors`（机器人点云）
和配套的带法线损失。Chamfer 那部分是从原始深度反投影出来的，跟渲染器无关。
所以 PyBullet 版换的还是同样两样东西，优化器、Chamfer、步数、学习率全部照搬。

下面是**端到端**对比：A 走 `align_A → joint_A`，B 走 `align_B → joint_B`，
各自吃自己上一阶段的结果，这才是"B 替换 A"的真实含义。

In [ ]:
import torch.optim as optim

def joint_pybullet(scene_constants, prev_state, pb, lr=0.001, n_steps=500,
                   robot_weight=1.0, chamfer_n_points=2000):
    cam1, cam2 = EXT_CAMS
    T_ee_all = scene_constants["robot"]["T_ee_base_all"]

    def robot_cloud(cam):
        is_wrist = (cam == WRIST)
        K_r = scene_constants["camera"][cam]["K_mat"]
        T_base = prev_state[cam]["base_extrinsic"]
        X, obs = [], []
        for t in range(N_FRAMES):
            pb.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                 scene_constants["robot"]["gripper_positions"][t])
            d_obs = scene_constants["camera"][cam]["raw_depth"][t].astype(np.float32)
            if is_wrist:
                pts = core.physics.get_foreground_gripper_points(
                    T_ee_all[t] @ T_base, K_r, d_obs, pb, device)
                if pts is None:
                    continue
                X.append(torch.tensor((T_base @ pts)[:3, :].T,
                                      dtype=torch.float32, device=device))
            else:
                pts = core.physics.get_foreground_robot_points(
                    T_base, K_r, d_obs, pb, device)
                if pts is None:
                    continue
                X.append(pts)
            obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None])
        return torch.stack(X), torch.stack(obs)

    t0 = time.perf_counter()
    X1, obs1 = robot_cloud(cam1)
    X2, obs2 = robot_cloud(cam2)
    Xw, obsw = robot_cloud(WRIST)

    Pc1, Pc2, Pcw, Tee = [], [], [], []
    for t in range(N_FRAMES):
        pcs = [
            compute_extrinsics.get_cam_points_local_t(
                t, scene_constants["camera"][c], device, chamfer_n_points)
            for c in (cam1, cam2, WRIST)]
        if all(pc is not None for pc in pcs):
            Pc1.append(pcs[0]); Pc2.append(pcs[1]); Pcw.append(pcs[2])
            Tee.append(torch.tensor(T_ee_all[t], dtype=torch.float32, device=device))
    Pc1, Pc2, Pcw, Tee = (torch.stack(x) for x in (Pc1, Pc2, Pcw, Tee))
    t_extract = time.perf_counter() - t0

    Kt = {c: torch.tensor(scene_constants["camera"][c]["K_mat"],
                          dtype=torch.float32, device=device)
          for c in (cam1, cam2, WRIST)}
    Tin = {c: torch.tensor(prev_state[c]["base_extrinsic"],
                           dtype=torch.float32, device=device)
           for c in (cam1, cam2, WRIST)}
    d1 = torch.zeros(6, requires_grad=True, device=device)
    d2 = torch.zeros(6, requires_grad=True, device=device)
    dhe = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d1, d2, dhe], lr=lr)

    t0 = time.perf_counter()
    for step in range(n_steps):
        optimizer.zero_grad()
        T1 = Tin[cam1] @ core.geometry.make_T(d1, device)
        T2 = Tin[cam2] @ core.geometry.make_T(d2, device)
        Tee_opt = Tin[WRIST] @ core.geometry.make_T(dhe, device)

        bc1 = (T1 @ Pc1)[:, :3, :].transpose(1, 2)
        bc2 = (T2 @ Pc2)[:, :3, :].transpose(1, 2)
        bcw = torch.bmm(Tee @ Tee_opt, Pcw)[:, :3, :].transpose(1, 2)
        l12, o12 = compute_extrinsics.batched_chamfer_distance(bc1, bc2, device)
        l1w, o1w = compute_extrinsics.batched_chamfer_distance(bc1, bcw, device)
        l2w, o2w = compute_extrinsics.batched_chamfer_distance(bc2, bcw, device)
        loss_chamfer = l12 + l1w + l2w

        r1 = core.physics.compute_robot_loss_batched(X1, T1, Kt[cam1], obs1)
        r2 = core.physics.compute_robot_loss_batched(X2, T2, Kt[cam2], obs2)
        rw = core.physics.compute_wrist_loss_batched(Xw, Tee_opt, Kt[WRIST], obsw)
        (loss_chamfer + robot_weight * (r1 + r2 + rw)).backward()
        optimizer.step()

        if step % 100 == 0 or step == n_steps - 1:
            print(f"    step {step:03d} | chamfer {loss_chamfer.item():.4f} | "
                  f"robot {r1.item():.4f}/{r2.item():.4f}/{rw.item():.4f} | "
                  f"bg {(o12 + o1w + o2w) / 3 * 100:.1f}%")
    t_opt = time.perf_counter() - t0

    with torch.no_grad():
        out = {}
        for cam, d in ((cam1, d1), (cam2, d2), (WRIST, dhe)):
            T = (Tin[cam] @ core.geometry.make_T(d, device)).cpu().numpy()
            out[cam] = {"base_extrinsic": T,
                        "extrinsics": (T_ee_all @ T if cam == WRIST
                                       else np.tile(T, (N_FRAMES, 1, 1)))}
    print(f"    取点云 {t_extract:.1f} s + 优化 {t_opt:.1f} s")
    return out, t_extract + t_opt

In [ ]:
print("B: pybullet joint")
state_B3, T_B3 = joint_pybullet(scene_constants, state_B, r_gpu)

print("\nA: yourdfpy joint")
t0 = time.perf_counter()
state_A3 = compute_extrinsics.global_joint_alignment(
    scene_constants, state_A, tensor_renderer)
T_A3 = time.perf_counter() - t0
print(f"\njoint 耗时: A {T_A3:.1f} s   B {T_B3:.1f} s")

print("\njoint2 (lr=1e-4, robot_weight=0.1)")
state_A4 = compute_extrinsics.global_joint_alignment(
    scene_constants, state_A3, tensor_renderer, lr=0.0001, robot_weight=0.1)
state_B4, _ = joint_pybullet(scene_constants, state_B3, r_gpu,
                             lr=0.0001, robot_weight=0.1)

In [ ]:
STAGES = [("起点（旧流水线产出）", scene_state_ds),
          ("A: align", state_A), ("A: +joint", state_A3), ("A: +joint2", state_A4),
          ("B: align", state_B), ("B: +joint", state_B3), ("B: +joint2", state_B4)]

SCORES = {}
for name, st in STAGES:
    np.random.seed(SEED + 9000)
    torch.manual_seed(SEED + 9000)
    SCORES[name] = compute_metrics.evaluate_extrinsics(
        scene_constants, st, device, pb_renderer=r_gpu)

keys = [("robot_loss_cam1", "robot cam1"), ("robot_loss_cam2", "robot cam2"),
        ("robot_loss_wrist", "robot wrist"), ("chamfer_total", "chamfer"),
        ("bg_overlap_pct", "bg overlap %")]
print(f"{'':<16}" + "".join(f"{lab:>14}" for _, lab in keys))
for name, m in SCORES.items():
    print(f"{name:<16}" + "".join(f"{m.get(k, float('nan')):>14.4f}" for k, _ in keys))

print("\n最终两条路解出来的外参差多少 (A:+joint vs B:+joint):")
for cam in scene_constants["camera"]:
    dT = np.linalg.inv(state_A3[cam]["base_extrinsic"]) @ state_B3[cam]["base_extrinsic"]
    shift = np.linalg.norm(dT[:3, 3]) * 1000
    rot = np.degrees(np.arccos(np.clip((np.trace(dT[:3, :3]) - 1) / 2, -1, 1)))
    print(f"  {cam:<12} 平移 {shift:7.2f} mm   旋转 {rot:6.3f}°")

In [ ]:
keys_ = [n for n, _ in STAGES]
labels = ["prev pipeline"] + keys_[1:]
colours = [C_THIRD, C_CPU, C_CPU, C_CPU, C_GPU, C_GPU, C_GPU]
panels = [
    ("robot depth loss, cam1  (lower is better)",
     [SCORES[n].get("robot_loss_cam1", np.nan) for n in keys_], "%.4f"),
    ("robot depth loss, wrist  (lower is better)",
     [SCORES[n].get("robot_loss_wrist", np.nan) for n in keys_], "%.4f"),
    ("chamfer total  (lower is better)",
     [SCORES[n].get("chamfer_total", np.nan) for n in keys_], "%.4f"),
]
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2), facecolor=SURFACE)
for ax, (title, vals, fmt) in zip(axes, panels):
    y = np.arange(len(labels))[::-1]
    ax.barh(y, vals, height=0.5, color=colours, zorder=3)
    span = max(v for v in vals if np.isfinite(v))
    for yi, v in zip(y, vals):
        if np.isfinite(v):
            ax.text(v + span * 0.03, yi, fmt % v, va="center", fontsize=9, color=INK)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9, color=MUTED)
    ax.set_title(title, fontsize=9.5, color=INK, loc="left", pad=10)
    ax.set_xlim(0, span * 1.28)
    ax.set_facecolor(SURFACE); ax.xaxis.set_visible(False)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(GRID); ax.tick_params(length=0)
fig.tight_layout()
plt.show()

## 6. 当年为什么慢，现在还慢不慢

方法 B 上面用的是 `gpu=True` + 半分辨率。把这两条都退回去，就是 v50 那份代码的处境——
它的 `PyBulletRenderer_Robotiq` 里写的是 `find_spec('eglRendererPlugin')`，
一个不存在的模块名，所以插件从来没加载过，全程 CPU 全分辨率。

下面按 v50 的循环规模（Stage 2 每台相机 2 个基底 × OUTER 5 轮，加上 Stage 3/4 各一遍预计算）
把渲染预算算出来。

In [ ]:
MS_CPU_FULL = MS_CPU
MS_GPU_HALF = SWEEP[0.5][1]
FULL_FRAMES = N_FRAMES

renders = FULL_FRAMES * (2 * 2 * 5 + 5 + 3 + 3)
print(f"一个 episode（{FULL_FRAMES} 帧，不抽帧）的渲染次数: {renders}")
print(f"{'配置':<34}{'每帧':>10}{'总计':>12}")
for label, ms in (("v50 当年: CPU + 全分辨率", MS_CPU_FULL),
                  ("只修插件名: GPU + 全分辨率", MS_GPU),
                  ("(未采用) GPU + 0.5x", MS_GPU_HALF)):
    total = renders * ms / 1000
    print(f"{label:<34}{ms:>8.1f}ms{total / 60:>10.1f} min")
print(f"\n端到端提速（流水线实际用的全分辨率）: {MS_CPU_FULL / MS_GPU:.0f}x")

## 结论

三步逐条对照，都在真实 episode 上量过。

**第 1 步 —— 过。** 三台相机、12 帧真实位姿，`gpu=True` 与 CPU 的 mask IoU **0.994–0.995**
（比合成 pose 的 0.983 还高，因为机器人在画面里占 16–20%，边界像素占比更小），
表面深度中位差 **0.8–1.1e-4 m**。速度 **222 ms → 28 ms，7.8x**——比合成场景的 4.4 倍更大，
因为 CPU 光栅化器的开销随画面里的三角形数量涨，EGL 基本持平。

保留意见：某台相机上"光滑表面"像素的**最大**差达到 7.2e-2 m。中位数是 0.08 mm，
所以是极少数细结构像素上的判定分歧，但说明 3x3 邻域判据没能完全隔离边界效应。

**第 2 步 —— 过，但最后没要。** 这个 episode 降到 160x90 都还有 2592 个机器人像素。
点云等价性用"到全分辨率点云的最近邻距离"量：scale 1.0 自己对自己是 **1.92 mm**
（两次独立采样 2000 点的噪声地板），0.5x 是 **2.08 mm**，0.125x 才涨到 2.80 mm——
降到一半，误差还埋在采样噪声里。耗时 28.05 → 7.74 ms。

**即便如此，流水线里没有保留这个旋钮。** `compute_extrinsics` 一律全分辨率渲染：
省下的 20 ms/帧在整条流水线里换不来多少，而代价是机器人在画面里小的 episode
会整段取不到 2000 个点，joint 直接抛错。同理，这个 notebook 也不再抽帧
（原来的 `FRAME_STRIDE=4`），跑的就是 `compute_extrinsics` 跑的那批帧。
下面第 3 步的表是**抽帧 4、0.5x 渲染那一版跑出来的**（表头的 127/181/195 帧即由此而来）；
结论的方向不受影响，但绝对数字对不上现在的代码。

**第 3 步 —— B 改善外参，A 基本回到原地；joint2 两条路都做坏。**

单个 episode 上的结论会误导，所以下面是**三个 episode × 三个种子的均值**，
而且是**配对评分**——`evaluate_extrinsics` 每次调用都会重新随机抽 Chamfer 点云，
所以每个 state 打分前都把 RNG 重置到同一个值，七个 state 看到的是同一批点。
（指标代码本身对所有 state 完全相同：它跳哪些帧只取决于原始深度，
跟被评的外参无关。）

`↓` = 比初始好 3% 以上，`↑` = 差 3% 以上，`·` = 持平。

| AUTOLab (127 帧) | rob cam1 | rob cam2 | rob wrist | chamfer | bg ovl% |
|---|---|---|---|---|---|
| 起点（旧流水线产出） | 0.0210 | 0.0159 | 0.0180 | 0.0862 | 51.59 |
| A: align | 0.0171 ↓ | 0.0145 ↓ | 0.0143 ↓ | 0.0924 ↑ | 51.26 |
| A: +joint | 0.0204 · | 0.0153 ↓ | 0.0159 ↓ | 0.0863 · | 51.59 |
| A: +joint2 | 0.0397 ↑ | 0.0352 ↑ | 0.0128 ↓ | 0.0857 · | 52.86 |
| B: align | 0.0181 ↓ | 0.0172 ↑ | 0.0096 ↓ | 0.0921 ↑ | 50.96 |
| **B: +joint** | **0.0179** ↓ | 0.0157 · | **0.0084** ↓ | 0.0881 · | 51.09 |
| B: +joint2 | 0.0258 ↑ | 0.0155 · | 0.0095 ↓ | 0.0860 · | 51.83 |

| TRI (181 帧) | rob cam1 | rob cam2 | rob wrist | chamfer | bg ovl% |
|---|---|---|---|---|---|
| 起点（旧流水线产出） | 0.0103 | 0.0238 | 0.0108 | 0.1109 | 45.32 |
| A: align | 0.0102 · | 0.0208 ↓ | 0.0147 ↑ | 0.1225 ↑ | 44.67 |
| A: +joint | 0.0103 · | 0.0272 ↑ | 0.0121 ↑ | 0.1108 · | 45.23 |
| A: +joint2 | 0.0168 ↑ | 0.0523 ↑ | 0.0105 · | 0.1088 · | 45.94 |
| B: align | 0.0107 ↑ | 0.0223 ↓ | 0.0097 ↓ | 0.1286 ↑ | 43.54 |
| **B: +joint** | 0.0112 ↑ | **0.0227** ↓ | **0.0097** ↓ | 0.1115 · | 45.11 |
| B: +joint2 | 0.0124 ↑ | 0.0252 ↑ | 0.0100 ↓ | 0.1083 · | 45.44 |

| WEIRD (195 帧) | rob cam1 | rob cam2 | rob wrist | chamfer | bg ovl% |
|---|---|---|---|---|---|
| 起点（旧流水线产出） | 0.0107 | 0.1436 | 0.0171 | 0.0881 | 21.76 |
| A: align | 0.0106 · | 0.2050 ↑ | 0.0160 ↓ | 0.0993 ↑ | 21.06 |
| A: +joint | 0.0107 · | 0.2685 ↑ | 0.0188 ↑ | 0.1417 ↑ | 23.49 |
| A: +joint2 | 0.0200 ↑ | 0.3565 ↑ | 0.0261 ↑ | 0.1433 ↑ | 26.42 |
| B: align | 0.0108 · | 0.0999 ↓ | 0.0089 ↓ | 0.0987 ↑ | 19.21 |
| **B: +joint** | 0.0109 · | **0.1059** ↓ | **0.0095** ↓ | **0.0843** ↓ | 20.84 |
| B: +joint2 | 0.0111 ↑ | 0.1083 ↓ | 0.0095 ↓ | 0.0841 ↓ | 21.17 |

三条跨 episode 一致的结论：

**先说清楚"起点"是什么。** 上面所有表的第一行是 `load_extrinsics(...)` 读出来的，
那是 **Stage 2 自己上一次的产出**（`gs://dm-tapnet/tmp/droid/extrinsics`），
**不是** DROID 数据集自带的外参。数据集那份在元数据库里，是 `init_camera_states`
读的东西，也是 `compute_extrinsics.py` 真正的起点。两者差很多：

| | cam1 | cam2 | wrist | chamfer | bg% |
|---|---|---|---|---|---|
| AUTOLab 数据集元数据 | 0.0223 | 0.0358 | 0.0223 | 0.0966 | 51.96 |
| AUTOLab 旧流水线产出（本表起点） | 0.0211 | 0.0158 | 0.0170 | 0.0865 | 51.66 |
| TRI 数据集元数据 | 0.0201 | 0.1253 | 0.0180 | 0.1202 | 45.39 |
| TRI 旧流水线产出（本表起点） | 0.0099 | 0.0239 | 0.0103 | 0.1113 | 45.46 |
| WEIRD 数据集元数据 | 0.0291 | 0.1513 | 0.0135 | 0.1549 | 22.34 |
| WEIRD 旧流水线产出（本表起点） | 0.0107 | 0.1438 | 0.0171 | 0.0872 | 21.73 |

这对下面的读法有两个影响。一是"相对起点有没有改善"衡量的是
**有没有比旧流水线的产出更好**，不是有没有比数据集外参更好。
二是那个起点**正是 A 这套算法自己的不动点**——同一个目标函数从自己的解出发，
本来就该待着不动。所以让 A/B 都从这里起跑是**偏袒 A** 的设置；
B 还能走开并把腕部改善近一倍，这个差距只会被低估。

**1. `B: +joint` 是三个 episode 上最好的配置。** 腕部三个全改善（0.0180→0.0084 /
0.0108→0.0097 / 0.0171→0.0095），而 A 在 TRI 和 WEIRD 上腕部**比初始还差**。

**2. A 走完 joint 基本回到起点。** chamfer 在 AUTOLab 从 0.0862 出发回到 0.0863，
TRI 从 0.1109 回到 0.1108；量外参也证实——A:+joint 离初始只有 1.6–4.5 mm，
而它自己的 align 曾走出 3–9 mm。**joint 的 Chamfer 项把 A 的 align 成果抹掉了。**
所以"A 的 chamfer 更好"不是优化得更好，**是它没怎么动**——初始外参本来就接近 chamfer 最优。
B 站得住：走出 5–11 mm，joint 后仍在 2.6–6.1 mm。

**3. joint2 两条路都做坏，A 坏得多。** `robot_weight` 从 1.0 降到 0.1 之后 Chamfer 项彻底压过
robot 项：AUTOLab 上 A 的 cam1 从 0.0204 崩到 0.0397（比初始 0.0210 还差近一倍），
TRI 上 cam2 从 0.0272 崩到 0.0523，WEIRD 上从 0.2685 崩到 0.3565。
换来的 chamfer 改善只有 0.002 上下。B 也退，但退得少得多（AUTOLab cam1 0.0179→0.0258）。
**这是关于流水线本身的结论，跟渲染器无关——仓库只跑到 joint 是对的。**

**WEIRD 最有信息量**：它的 cam2 起点就坏（0.1436），A 把它越推越远
（外参移 209 mm → 398 mm，损失 0.1436 → 0.2685 → 0.3565），B 把它拉回来
（移 143 → 121 mm，损失 0.1059）。**而且 WEIRD 上 B 连 chamfer 都赢**
（0.0843 vs 0.1417）——这条很重要，因为 chamfer 是不与 B 共用渲染器的指标之一。

**bg overlap**（两朵点云互相 5 cm 以内的点占比）不能单独当质量排序。
WEIRD 上 A:+joint2 的 bg 最高（26.42），chamfer 却最差（0.1433）——
落在截断内的点更多，但平均距离更大。要和 chamfer 一起读。

**耗时**（0.5x 渲染那一版）：align A 14.0 s / B 12.6 s，joint A 31.7 s / B 28.7 s
（B 的 joint 里取点云只占 3.9 s，其余是两条路共有的 500 步 Chamfer）。打平。
**换 PyBullet 的理由不是快。**

**必须挑明的偏倚**：robot 那三列是 `evaluate_extrinsics` 用 PyBullet 渲染算的，
和 B 的点云来源共用一套机器；chamfer 和 bg overlap 不共用。
好消息是 WEIRD 上 B 在 chamfer 上也赢；坏消息是 AUTOLab 上它偏向 A（0.0863 vs 0.0881，
配对评分后这个差距依然在，不是采样噪声）。三个 episode 还不足以定论。

### 多 GPU 上的一个坑（已修）

日志里那行 `EGL device choice: -1 of 32.` 是插件在说"没人指定设备，我自己挑"——
它挑的永远是 **GPU 0**，而且**只认 `EGL_VISIBLE_DEVICES`，不看 `CUDA_VISIBLE_DEVICES`**。

单进程无所谓，但 `run_parallel.sh` 是用 `CUDA_VISIBLE_DEVICES={}` 给每个 worker 分卡的：
批处理一开 `gpu=True`，16 个 worker 的 EGL 上下文会全挤在 GPU 0，而张量分散在 16 张卡上。

`core/physics._load_egl` 现在会在调用方没指定时把 `EGL_VISIBLE_DEVICES` 对齐到单卡的
`CUDA_VISIBLE_DEVICES`。四个 worker 分别 pin 到 GPU 2/5/9/13 实测，显存确实落在各自卡上
（每个上下文约 285 MiB，所以共享一张卡的瓶颈是光栅化吞吐不是显存）。
调用方自己设了 `EGL_VISIBLE_DEVICES`、或 pin 的是多卡列表 / UUID 时，它不插手。

### 还需要做什么

1. **更多 episode**。第 3 步现在有三个（AUTOLab / TRI / WEIRD），但第 1、2 步仍只有
   AUTOLab 一个。而且三个 episode 里已经出现了截然不同的行为（WEIRD 的 cam2 失控），
   要下"换掉 yourdfpy"的结论得跑 `episodes_eval50.txt` 那批。
2. **一把不共用渲染器的尺子**。robot 三列的裁判和 B 共用 PyBullet。
   要么用人工标注，要么用重投影到 RGB 的边缘对齐这类完全独立的指标。
3. **WEIRD 的 cam2 为什么会失控**。起点 0.1435 就不正常，两条路都把它移了十几厘米。
   这可能是深度质量问题而不是外参问题，值得先看一眼那台相机的原始深度。
4. **joint 为什么会抹掉 A 的 align 成果**。`robot_weight` 默认 1.0，
   Chamfer 是三项之和而 robot 是三项之和，量纲上 Chamfer 可能压过 robot 项——
   调权重也许能让 A 站住，那样对照才公平。
5. **`compute_tracks` 端到端影响**没测。第 1 步证明了单帧 mask IoU 0.994，
   但 0.6% 的边界像素分歧会不会累积到轨迹上，要跑完整 Stage 3 才知道。
6. 第 1 步那个 7.2e-2 m 的最大差值得单独看一眼是哪个像素。

在 1 和 2 之前，**不建议翻 `gpu=` 的默认值，也不建议动 `compute_extrinsics`**。